In [3]:
# ============================================================
# INGEST RACES — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

In [4]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_LANDING
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:07:38 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:07:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:07:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:07:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [ ]:
# -----------------------------------------
# 2. IMPORT HELPER FUNCTIONS
# -----------------------------------------
try:
    add_ingestion_metadata
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    helper_path = "/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb"
    with open(helper_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

✔ Bronze helper functions loaded


In [6]:
# -----------------------------------------
# 3. Define source + target paths
# -----------------------------------------
source_file = f"{BRONZE_LANDING}/races.csv"
target_folder = f"{BRONZE_ROOT}/races"
target_file = f"{target_folder}/races.parquet"

os.makedirs(target_folder, exist_ok=True)

print("Source:", source_file)
print("Target:", target_file)

Source: /Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/races.csv
Target: /Users/manoharazalki/F1-ANALYTICS/data/bronze/races/races.parquet


In [7]:
# -----------------------------------------
# 4. Define schema (same as Databricks)
# -----------------------------------------
races_schema = StructType([
    StructField('season', IntegerType(), True),
    StructField('round', IntegerType(), True),
    StructField('url', StringType(), True),
    StructField('raceName', StringType(), True),
    StructField('date', DateType(), True),
    StructField('circuitId', StringType(), True)
])

In [8]:
# -----------------------------------------
# 5. Read CSV
# -----------------------------------------
races_df = (
    spark.read
        .format("csv")
        .option("header", True)
        .option("mode", "FAILFAST")
        .schema(races_schema)
        .load(source_file)
)

print("✔ Races file read successfully")
races_df.show(5)

✔ Races file read successfully
+------+-----+--------------------+------------------+----------+------------+
|season|round|                 url|          raceName|      date|   circuitId|
+------+-----+--------------------+------------------+----------+------------+
|  1950|    1|https://en.wikipe...|british grand prix|1950-05-13| silverstone|
|  1950|    2|https://en.wikipe...| monaco grand prix|1950-05-21|      monaco|
|  1950|    3|https://en.wikipe...|  indianapolis 500|1950-05-30|indianapolis|
|  1950|    4|https://en.wikipe...|  swiss grand prix|1950-06-04|  bremgarten|
|  1950|    5|https://en.wikipe...|belgian grand prix|1950-06-18|         spa|
+------+-----+--------------------+------------------+----------+------------+
only showing top 5 rows


In [9]:
# -----------------------------------------
# 6. Add metadata
# -----------------------------------------
races_final_df = add_ingestion_metadata(races_df, source_file)

print("✔ Metadata added")
races_final_df.show(5)

✔ Metadata added
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+
|season|round|                 url|          raceName|      date|   circuitId|    ingest_timestamp|         source_file|
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+
|  1950|    1|https://en.wikipe...|british grand prix|1950-05-13| silverstone|2026-06-08 23:07:...|/Users/manoharaza...|
|  1950|    2|https://en.wikipe...| monaco grand prix|1950-05-21|      monaco|2026-06-08 23:07:...|/Users/manoharaza...|
|  1950|    3|https://en.wikipe...|  indianapolis 500|1950-05-30|indianapolis|2026-06-08 23:07:...|/Users/manoharaza...|
|  1950|    4|https://en.wikipe...|  swiss grand prix|1950-06-04|  bremgarten|2026-06-08 23:07:...|/Users/manoharaza...|
|  1950|    5|https://en.wikipe...|belgian grand prix|1950-06-18|         spa|2026-06-08 23:07:...|/Users/manoharaza...|
+------+-----+-

In [10]:
# -----------------------------------------
# 7. Write to Bronze (Option C)
# -----------------------------------------
(
    races_final_df
        .write
        .mode("overwrite")
        .parquet(target_file)
)

print(f"✔ Races Bronze written to: {target_file}")

✔ Races Bronze written to: /Users/manoharazalki/F1-ANALYTICS/data/bronze/races/races.parquet


In [11]:
# -----------------------------------------
# 8. Read back for validation
# -----------------------------------------
spark.read.parquet(target_file).show(5)

+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+
|season|round|                 url|          raceName|      date|   circuitId|    ingest_timestamp|         source_file|
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+
|  1950|    1|https://en.wikipe...|british grand prix|1950-05-13| silverstone|2026-06-08 23:07:...|/Users/manoharaza...|
|  1950|    2|https://en.wikipe...| monaco grand prix|1950-05-21|      monaco|2026-06-08 23:07:...|/Users/manoharaza...|
|  1950|    3|https://en.wikipe...|  indianapolis 500|1950-05-30|indianapolis|2026-06-08 23:07:...|/Users/manoharaza...|
|  1950|    4|https://en.wikipe...|  swiss grand prix|1950-06-04|  bremgarten|2026-06-08 23:07:...|/Users/manoharaza...|
|  1950|    5|https://en.wikipe...|belgian grand prix|1950-06-18|         spa|2026-06-08 23:07:...|/Users/manoharaza...|
+------+-----+------------------